In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"

from huggingface_hub import snapshot_download

local_ast_path = snapshot_download(
    repo_id="MIT/ast-finetuned-audioset-10-10-0.4593",
    local_dir="/root/autodl-tmp/official_ast_model",
    local_dir_use_symlinks=False,
    resume_download=True
)

print("Downloaded to:", local_ast_path)

/root/miniconda3/lib/python3.8/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/root/miniconda3/lib/python3.8/site-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Downloaded to: /root/autodl-tmp/official_ast_model


In [3]:
from transformers import ASTFeatureExtractor, ASTModel
import torch

local_ast_path = "/root/autodl-tmp/official_ast_model"
device = "cuda" if torch.cuda.is_available() else "cpu"

feature_extractor = ASTFeatureExtractor.from_pretrained(
    local_ast_path,
    local_files_only=True
)

ast_model = ASTModel.from_pretrained(
    local_ast_path,
    local_files_only=True
)

ast_model.to(device)
ast_model.eval()

print("Official AST loaded locally on:", device)

Official AST loaded locally on: cuda


In [2]:
# ============================================================
# Official Pretrained AST Baseline for YMIR
# Separate pipeline for fair comparison with MFCC20, MFCC40,
# and Mel-spectrogram based models.
# ============================================================

# If you are using Jupyter, uncomment this line if packages are missing:
# import sys
# !{sys.executable} -m pip install torch torchaudio transformers librosa scikit-learn pandas -q

import os
import csv
import numpy as np
import pandas as pd
import librosa
import torch

from transformers import ASTFeatureExtractor, ASTModel

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# Settings
# ============================================================

DATA_PATH = "/root/YMED/"
GENRES = ["adeni", "hadrami", "lahji", "sanaai", "tihami"]

# Official AST uses 16 kHz audio
SR = 16000

FILE_DURATION = 30.0
SEGMENTS = 5
SEGMENT_DURATION = 6.0

TEST_SIZE = 0.2
RANDOM_STATE = 42

AST_MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

SAVE_DIR = "/root/autodl-tmp/official_ast_results/"
EMB_DIR = os.path.join(SAVE_DIR, "embeddings")
META_DIR = os.path.join(SAVE_DIR, "metadata")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)


# ============================================================
# Optional Hugging Face Mirror
# Use this if your server has connection issues with huggingface.co
# ============================================================

USE_HF_MIRROR = True

if USE_HF_MIRROR:
    os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
    os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
    os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"


# ============================================================
# 1. Create Official AST Model
# ============================================================

def create_official_ast_model(local_model_path=None, device=None):
    """
    Load official pretrained AST model.

    If local_model_path is provided, the model is loaded offline.
    Otherwise, it is downloaded from Hugging Face or HF mirror.

    Returns:
        feature_extractor, ast_model, device
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if local_model_path is not None:
        print("Loading Official AST from local path:", local_model_path)

        feature_extractor = ASTFeatureExtractor.from_pretrained(
            local_model_path,
            local_files_only=True
        )

        ast_model = ASTModel.from_pretrained(
            local_model_path,
            local_files_only=True
        )

    else:
        print("Loading Official AST from:", AST_MODEL_NAME)

        feature_extractor = ASTFeatureExtractor.from_pretrained(AST_MODEL_NAME)
        ast_model = ASTModel.from_pretrained(AST_MODEL_NAME)

    ast_model.to(device)
    ast_model.eval()

    print("=" * 70)
    print("Official pretrained AST loaded successfully.")
    print("Model:", AST_MODEL_NAME if local_model_path is None else local_model_path)
    print("Device:", device)
    print("=" * 70)

    return feature_extractor, ast_model, device


# ============================================================
# 2. Collect Original Files
# ============================================================

def collect_original_files(data_path, genres):
    """
    Collect original WAV files before segmentation.
    This is necessary to prevent data leakage.
    """

    file_paths = []
    labels = []

    for idx, genre in enumerate(genres):
        genre_dir = os.path.join(data_path, genre)

        if not os.path.isdir(genre_dir):
            raise FileNotFoundError(f"Genre folder not found: {genre_dir}")

        wav_files = sorted([
            f for f in os.listdir(genre_dir)
            if f.lower().endswith(".wav")
        ])

        print(f"{genre}: {len(wav_files)} original WAV files")

        for filename in wav_files:
            file_paths.append(os.path.join(genre_dir, filename))
            labels.append(idx)

    return np.array(file_paths), np.array(labels)


# ============================================================
# 3. Audio Loading and Segmentation
# ============================================================

def load_audio_30s(file_path, sr=SR, duration=FILE_DURATION):
    """
    Load audio at 16 kHz and force it to exactly 30 seconds.
    Short files are zero-padded; long files are truncated.
    """

    target_len = int(sr * duration)

    audio, _ = librosa.load(
        file_path,
        sr=sr,
        mono=True
    )

    if len(audio) < target_len:
        audio = np.pad(
            audio,
            (0, target_len - len(audio)),
            mode="constant"
        )
    else:
        audio = audio[:target_len]

    return audio.astype(np.float32)


def extract_segments(audio, sr=SR):
    """
    Split one 30-second audio file into five non-overlapping 6-second segments.
    """

    segment_len = int(SEGMENT_DURATION * sr)
    segments = []

    for i in range(SEGMENTS):
        start = i * segment_len
        end = start + segment_len

        segment = audio[start:end]

        if len(segment) == segment_len:
            segments.append(segment.astype(np.float32))

    return segments


# ============================================================
# 4. Leakage Check
# ============================================================

def check_no_data_leakage(train_files, test_files):
    """
    Confirm that no original file appears in both training and testing sets.
    """

    train_base = set(os.path.basename(f) for f in train_files)
    test_base = set(os.path.basename(f) for f in test_files)

    overlap = train_base.intersection(test_base)

    if len(overlap) > 0:
        raise RuntimeError(f"Data leakage detected: {list(overlap)[:10]}")

    print("\nData leakage check PASSED.")
    print("No original 30-second recording appears in both train and test.")


# ============================================================
# 5. AST Embedding Extraction
# ============================================================

def extract_ast_embedding(audio_segment, feature_extractor, ast_model, device, sr=SR):
    """
    Extract CLS-token embedding from official AST for one raw audio segment.

    Input:
        audio_segment: np.ndarray, shape = (audio_samples,)

    Output:
        embedding: np.ndarray, shape = (768,)
    """

    audio_segment = audio_segment.astype(np.float32)

    inputs = feature_extractor(
        audio_segment,
        sampling_rate=sr,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = ast_model(**inputs)

    # CLS-token embedding
    embedding = outputs.last_hidden_state[:, 0, :]

    return embedding.detach().cpu().numpy()[0].astype(np.float32)


def extract_ast_embeddings_from_files(
    file_paths,
    labels,
    feature_extractor,
    ast_model,
    device,
    split_name="train"
):
    """
    Extract AST embeddings for all 6-second segments from a set of files.
    """

    X_embeddings = []
    y_out = []
    metadata = []

    for file_idx, (file_path, label) in enumerate(zip(file_paths, labels)):
        print(f"[{split_name}] {file_idx + 1}/{len(file_paths)}: {os.path.basename(file_path)}")

        try:
            audio = load_audio_30s(file_path)
            segments = extract_segments(audio)

            for segment_id, segment in enumerate(segments):
                embedding = extract_ast_embedding(
                    audio_segment=segment,
                    feature_extractor=feature_extractor,
                    ast_model=ast_model,
                    device=device,
                    sr=SR
                )

                X_embeddings.append(embedding)
                y_out.append(int(label))

                metadata.append({
                    "split": split_name,
                    "file_path": file_path,
                    "file_name": os.path.basename(file_path),
                    "segment_id": segment_id,
                    "label": int(label),
                    "genre": GENRES[int(label)]
                })

        except Exception as e:
            print(f"Skipping file due to error: {file_path}")
            print("Error:", e)

    return (
        np.array(X_embeddings, dtype=np.float32),
        np.array(y_out, dtype=np.int64),
        metadata
    )


# ============================================================
# 6. Save Metadata
# ============================================================

def save_metadata(metadata, save_path):
    """
    Save segment metadata for reproducibility and leakage checking.
    """

    if len(metadata) == 0:
        return

    keys = metadata[0].keys()

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(metadata)


# ============================================================
# 7. Train Classifier on AST Embeddings
# ============================================================

def train_ast_svm_classifier(X_train, y_train, X_test, y_test):
    """
    Train SVM classifier on official AST embeddings.
    """

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    classifier = SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        probability=True,
        random_state=RANDOM_STATE
    )

    print("\nTraining SVM classifier on official AST embeddings...")
    classifier.fit(X_train_scaled, y_train)

    y_pred = classifier.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print("\n" + "=" * 70)
    print("Official Pretrained AST + SVM Results")
    print("=" * 70)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=GENRES, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    result = {
        "Model": "Official_AST_Pretrained",
        "Input": "Raw waveform",
        "Feature": "AST embedding",
        "Classifier": "SVM-RBF",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

    return classifier, scaler, y_pred, result


# ============================================================
# 8. Run Official AST Experiment
# ============================================================

def run_official_ast_experiment(local_model_path=None):
    """
    Complete AST experiment:
        original WAV split
        segmentation inside each split
        AST embedding extraction
        SVM classification
        result saving
    """

    print("=" * 70)
    print("Running Official Pretrained AST Baseline")
    print("=" * 70)

    # --------------------------------------------------------
    # Collect original files
    # --------------------------------------------------------
    file_paths, labels = collect_original_files(DATA_PATH, GENRES)

    print("\nTotal original files:", len(file_paths))

    # --------------------------------------------------------
    # Track-level split before segmentation
    # --------------------------------------------------------
    train_files, test_files, y_train_files, y_test_files = train_test_split(
        file_paths,
        labels,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=labels
    )

    print("\nOriginal-file split:")
    print("Train original files:", len(train_files))
    print("Test original files :", len(test_files))

    check_no_data_leakage(train_files, test_files)

    # --------------------------------------------------------
    # Save original split for reproducibility
    # --------------------------------------------------------
    split_csv = os.path.join(META_DIR, "official_ast_original_split.csv")

    with open(split_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["split", "file_path", "label", "genre"])

        for fp, lb in zip(train_files, y_train_files):
            writer.writerow(["train", fp, int(lb), GENRES[int(lb)]])

        for fp, lb in zip(test_files, y_test_files):
            writer.writerow(["test", fp, int(lb), GENRES[int(lb)]])

    print("Original split saved to:", split_csv)

    # --------------------------------------------------------
    # Load official AST
    # --------------------------------------------------------
    feature_extractor, ast_model, device = create_official_ast_model(
        local_model_path=local_model_path
    )

    # --------------------------------------------------------
    # Extract or load embeddings
    # --------------------------------------------------------
    train_emb_path = os.path.join(EMB_DIR, "X_train_ast_official.npy")
    train_y_path = os.path.join(EMB_DIR, "y_train_ast_official.npy")
    test_emb_path = os.path.join(EMB_DIR, "X_test_ast_official.npy")
    test_y_path = os.path.join(EMB_DIR, "y_test_ast_official.npy")

    if (
        os.path.exists(train_emb_path)
        and os.path.exists(train_y_path)
        and os.path.exists(test_emb_path)
        and os.path.exists(test_y_path)
    ):
        print("\nLoading saved AST embeddings...")

        X_train_ast = np.load(train_emb_path)
        y_train_ast = np.load(train_y_path)
        X_test_ast = np.load(test_emb_path)
        y_test_ast = np.load(test_y_path)

    else:
        print("\nExtracting AST embeddings for training split...")
        X_train_ast, y_train_ast, train_metadata = extract_ast_embeddings_from_files(
            train_files,
            y_train_files,
            feature_extractor,
            ast_model,
            device,
            split_name="train"
        )

        print("\nExtracting AST embeddings for testing split...")
        X_test_ast, y_test_ast, test_metadata = extract_ast_embeddings_from_files(
            test_files,
            y_test_files,
            feature_extractor,
            ast_model,
            device,
            split_name="test"
        )

        np.save(train_emb_path, X_train_ast)
        np.save(train_y_path, y_train_ast)
        np.save(test_emb_path, X_test_ast)
        np.save(test_y_path, y_test_ast)

        save_metadata(
            train_metadata,
            os.path.join(META_DIR, "official_ast_train_metadata.csv")
        )

        save_metadata(
            test_metadata,
            os.path.join(META_DIR, "official_ast_test_metadata.csv")
        )

        print("\nAST embeddings saved to:", EMB_DIR)

    print("\nEmbedding shapes:")
    print("X_train_ast:", X_train_ast.shape)
    print("y_train_ast:", y_train_ast.shape)
    print("X_test_ast :", X_test_ast.shape)
    print("y_test_ast :", y_test_ast.shape)

    # --------------------------------------------------------
    # Train classifier and evaluate
    # --------------------------------------------------------
    classifier, scaler, y_pred, result = train_ast_svm_classifier(
        X_train_ast,
        y_train_ast,
        X_test_ast,
        y_test_ast
    )

    # --------------------------------------------------------
    # Save final result
    # --------------------------------------------------------
    result_csv = os.path.join(SAVE_DIR, "official_ast_result.csv")
    pd.DataFrame([result]).to_csv(result_csv, index=False)

    print("\nFinal AST result saved to:", result_csv)

    return {
        "classifier": classifier,
        "scaler": scaler,
        "y_pred": y_pred,
        "result": result,
        "X_train_ast": X_train_ast,
        "y_train_ast": y_train_ast,
        "X_test_ast": X_test_ast,
        "y_test_ast": y_test_ast
    }


# ============================================================
# 9. Execute
# ============================================================

# If you downloaded AST manually, put the folder path here:
# Example:
# local_model_path = "/root/autodl-tmp/official_ast_model"

local_model_path = None

ast_outputs = run_official_ast_experiment(
    local_model_path=local_model_path
)

print("\nOfficial AST comparison row:")
print(ast_outputs["result"])

Running Official Pretrained AST Baseline
adeni: 230 original WAV files
hadrami: 230 original WAV files
lahji: 230 original WAV files
sanaai: 230 original WAV files
tihami: 230 original WAV files

Total original files: 1150

Original-file split:
Train original files: 920
Test original files : 230

Data leakage check PASSED.
No original 30-second recording appears in both train and test.
Original split saved to: /root/autodl-tmp/official_ast_results/metadata/official_ast_original_split.csv
Loading Official AST from: MIT/ast-finetuned-audioset-10-10-0.4593
Official pretrained AST loaded successfully.
Model: MIT/ast-finetuned-audioset-10-10-0.4593
Device: cuda

Loading saved AST embeddings...

Embedding shapes:
X_train_ast: (4600, 768)
y_train_ast: (4600,)
X_test_ast : (1150, 768)
y_test_ast : (1150,)

Training SVM classifier on official AST embeddings...

Official Pretrained AST + SVM Results
Accuracy : 0.9574
Precision: 0.9578
Recall   : 0.9574
F1-score : 0.9572

Classification Report:
 